# Grouping Freesound recordings with k-means

Can audio descriptors group single notes by instrument? Each point represents one Freesound recording. K-means sees only numeric descriptors. After fitting, we compare its groups with the instrument names used to retrieve the recordings.

The setup cell reuses cached descriptor JSON files and downloads the collection when the cache is empty. The plots show where sounds lie, what distinguishes the cluster centers, and how each cluster mixes search categories.

In [ ]:
import os
from pathlib import Path
from typing import Any
import platform
import subprocess

import numpy as np
import json
import matplotlib
import matplotlib.pyplot as plt
import freesound as fs
from scipy.cluster.vq import kmeans, whiten

# Current API fields mapped to the existing descriptor-cache keys.
api_fields = {
    'spectral_centroid': 'lowlevel.spectral_centroid.mean',
    'dissonance': 'lowlevel.dissonance.mean',
    'log_attack_time': 'sfx.logattacktime.mean',
    'inharmonicity': 'sfx.inharmonicity.mean',
    'mfcc': 'lowlevel.mfcc.mean',
}
mfcc_key = api_fields['mfcc']
descriptorMapping = dict(enumerate([
    *(key for field, key in api_fields.items() if field != 'mfcc'),
    *(f'{mfcc_key}.{i}' for i in range(13)),
]))

## 1. Prepare the recordings

The example searches for single notes from cello, violin, bassoon, guitar, and trumpet. Search categories are useful for comparison and can include ambiguous or mislabeled recordings. Each sound needs a descriptor JSON file. Cached previews enable listening.

Set `DATA_DIR` to the cache directory, which contains category folders with one subfolder per sound. If the directory has no descriptor files, the setup cell uses your Freesound API key to populate it. Set `DOWNLOAD_SOUNDS = True` to rerun the download step for an existing cache.

In [ ]:
import requests
import time
import random

def download_sounds_freesound(queryText = "", tag=None, duration=None, API_Key = "", outputDir = "", topNResults = 5, featureExt = '.json', max_retries=4, retry_base_seconds=2.0):
  """
  Download Freesound previews and descriptor JSON files with retry support for transient API errors.
  """ 

  if queryText == "":
    print("\nProvide a query text to search for sounds")
    return -1

  if API_Key == "":
    print("\nYou need a valid Freesound API key. Apply at: www.freesound.org/apiv2/apply/")
    return -1

  if outputDir == "" or not os.path.exists(outputDir):
    print("\nPlease provide a valid output directory.")
    return -1

  fsClnt = fs.FreesoundClient()
  fsClnt.set_token(API_Key,"token")

  if duration and type(duration) == tuple:
    flt_dur = f" duration:[{str(duration[0])} TO {str(duration[1])}]"
  else:
    flt_dur = ""

  if tag and type(tag) == str:
    flt_tag = "tag:" + tag
  else:
    flt_tag = ""

  def _run_with_retries(callable_fn, action_name="request"):
    last_error = None
    for attempt in range(max_retries + 1):
      try:
        return callable_fn()
      except Exception as error:
        last_error = error
        message = str(error)
        retryable = any(code in message for code in ["504", "502", "503", "429", "timed out", "Gateway Time-out"])
        if (not retryable) or attempt >= max_retries:
          raise
        wait = retry_base_seconds * (2 ** attempt) + random.uniform(0, 0.5)
        print(f"{action_name} failed ({message}). Retrying in {wait:.1f}s [{attempt+1}/{max_retries}]...")
        time.sleep(wait)
    raise last_error

  page_size = 30
  if flt_tag + flt_dur != "":
    qRes = _run_with_retries(
      lambda: fsClnt.text_search(
        query=queryText,
        filter=flt_tag + flt_dur,
        sort="score",
        fields="id,name,previews,url",
        page_size=page_size
      ),
      action_name=f"text_search({queryText})"
    )
  else:
    qRes = _run_with_retries(
      lambda: fsClnt.text_search(
        query=queryText,
        sort="score",
        fields="id,name,previews,url",
        page_size=page_size
      ),
      action_name=f"text_search({queryText})"
    )

  outDir2 = Path(outputDir) / queryText
  outDir2.mkdir(exist_ok=True)

  def fetch_analysis(sound_id):
    response = requests.get(
      f"https://freesound.org/apiv2/sounds/{sound_id}/analysis/",
      headers={"Authorization": f"Token {API_Key}"},
      timeout=30,
    )
    response.raise_for_status()
    analysis = response.json()
    missing = [field for field in api_fields if analysis.get(field) is None]
    if missing:
      raise ValueError(f"Freesound analysis is missing: {', '.join(missing)}")
    features = {cache_key: [analysis[field]] for field, cache_key in api_fields.items()}
    mfcc = np.asarray(analysis['mfcc'], dtype=float)
    if mfcc.shape != (13,):
      raise ValueError(f"Expected 13 MFCCs, received shape {mfcc.shape}")
    scalars = [analysis[field] for field in api_fields if field != 'mfcc']
    if not np.all(np.isfinite(np.r_[scalars, mfcc])):
      raise ValueError("Freesound analysis contains non-finite descriptors")
    return features

  pageNo = 1
  sndCnt = 0
  indCnt = 0
  totalSnds = min(qRes.count, 200)

  downloadedSounds = []
  while(1):
    if indCnt >= totalSnds:
      print("Not enough results or too many failed downloads for this query.")
      break

    sound = qRes[indCnt - ((pageNo-1)*page_size)]
    outDir1 = outDir2 / str(sound.id)
    ftrPath = outDir1 / f"analysis{featureExt}"

    try:
      if not ftrPath.is_file():
        features = _run_with_retries(
          lambda: fetch_analysis(sound.id),
          action_name=f"fetch analysis for sound {sound.id}"
        )
        outDir1.mkdir(exist_ok=True)
        ftrPath.write_text(json.dumps(features), encoding="utf-8")

      sndCnt += 1
      downloadedSounds.append([str(sound.id), sound.url])
      mp3Path = outDir1 / f"{sound.id}.mp3"
      if not any(outDir1.glob("*.mp3")):
        try:
          _run_with_retries(
            lambda: fs.FSRequest.retrieve(sound.previews.preview_hq_mp3, fsClnt, str(mp3Path)),
            action_name=f"download preview for sound {sound.id}"
          )
        except Exception as error:
          mp3Path.unlink(missing_ok=True)
          print(f"Preview unavailable for sound {sound.id}: {error}")
    except Exception as error:
      print(f"Skipping descriptors for sound {sound.id}: {error}")

    indCnt += 1

    if indCnt % page_size == 0 and indCnt < totalSnds:
      qRes = _run_with_retries(
        lambda: qRes.next_page(),
        action_name=f"next_page({queryText})"
      )
      pageNo += 1

    if sndCnt >= topNResults:
      break

  with open(os.path.join(outDir2, queryText + '_SoundList.txt'), 'w') as fid:
    for elem in downloadedSounds:
      fid.write('\t'.join(elem) + '\n')

  print(f"Downloaded {sndCnt} sounds for '{queryText}'.")
  return sndCnt

In [ ]:
DATA_DIR = Path('descSounds')
DOWNLOAD_SOUNDS = False  # Set True to rerun downloads even with a populated cache.

def descriptor_files(directory, extension='.json'):
    return sorted(Path(directory).glob(f'*/*/*{extension}'))

if DOWNLOAD_SOUNDS or not descriptor_files(DATA_DIR):
    from dotenv import load_dotenv

    output_dir = DATA_DIR
    output_dir.mkdir(exist_ok=True)

    # Load API key from .env file (copy .env.example → .env and fill in your key)
    load_dotenv()
    api_key = os.environ.get("FREESOUND_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "FREESOUND_API_KEY not set. "
            "Set it in the environment or in a .env file at the repository root."
        )

    queries = ['cello', 'violin', 'bassoon', 'guitar', 'trumpet']
    download_summary = {}

    for query in queries:
        try:
            count = download_sounds_freesound(
                queryText=query,
                API_Key=api_key,
                outputDir=str(output_dir),
                topNResults=20,
                duration=(0, 5),
                tag='single-note',
                max_retries=4,
                retry_base_seconds=2.0,
            )
            download_summary[query] = count
        except Exception as error:
            download_summary[query] = f"ERROR: {error}"
            print(f"Failed for '{query}': {error}")

    print("\nDownload summary:")
    for query, result in download_summary.items():
        print(f"- {query}: {result}")

    if not descriptor_files(DATA_DIR):
        raise RuntimeError(
            f"No descriptors were downloaded to {DATA_DIR.resolve()}. "
            "Check the per-category download summary above."
        )
else:
    print(f"Using cached descriptors in {DATA_DIR.resolve()}")

## 2. Turn recordings into points

The descriptor matrix has one row per recording and one column per selected feature. Spectral centroid describes brightness, log attack time describes how quickly a sound starts, and MFCCs describe the spectral envelope. The feature vector includes 13 mean MFCCs alongside centroid, dissonance, log attack time, and inharmonicity.

K-means uses Euclidean distance. Before fitting, we divide each column by its standard deviation so differences are measured on comparable scales. Constant columns contribute zero to pairwise distances. K-means alternates between assigning each sound to its closest centroid and updating each centroid to the mean of its assigned sounds. The cluster numbers are arbitrary.

In [ ]:
def convFtrDict2List(ftrDict):
  """
  This function converts descriptor dictionary to an np.array. The order in the numpy array (indices) 
  are same as those mentioned in descriptorMapping dictionary.
  
  Input: 
    ftrDict (dict): dictionary containing descriptors downloaded from the freesound
  Output: 
    ftr (np.ndarray): Numpy array containing the descriptors for processing later on
  """
  ftr = []
  for key in range(len(descriptorMapping.keys())):
    try:
      ftrName, ind = '.'.join(descriptorMapping[key].split('.')[:-1]), int(descriptorMapping[key].split('.')[-1])
      ftr.append(ftrDict[ftrName][0][ind])
    except:
      ftr.append(ftrDict[descriptorMapping[key]][0])
  return np.array(ftr)

def fetchDataDetails(inputDir, descExt = '.json'):
  """
  This function is used by other functions to obtain the information regarding the directory structure 
  and the location of descriptor files for each sound 
  """
  dataDetails = {}
  for file in descriptor_files(inputDir, descExt):
    category = file.parent.parent.name
    sound_id = file.parent.name
    with file.open(encoding='utf-8') as stream:
      features = json.load(stream)
    dataDetails.setdefault(category, {})[sound_id] = {'file': file.name, 'feature': features}
  return dataDetails

def cluster_sounds(targetDir, nCluster = -1, descInput=[]):
  """
  This function clusters all the sounds in targetDir using kmeans clustering.
  
  Input:
    targetDir (string): Directory where sound descriptors are stored (all the sounds in this 
                        directory will be used for clustering)
    nCluster (int): Number of clusters to be used for kmeans clustering.
    descInput (list) : List of indices of the descriptors to be used for similarity/distance 
                       computation (see descriptorMapping)
  Output:
    Prints the class of each cluster (computed by a majority vote), number of sounds in each 
    cluster and information (sound-id, sound-class and classification decision) of the sounds 
    in each cluster. Optionally, you can uncomment the return statement to return the same data.
  """
  
  dataDetails = fetchDataDetails(targetDir)
  
  ftrArr = []
  infoArr = []
  
  if nCluster ==-1:
    nCluster = len(dataDetails.keys())
  for cname in dataDetails.keys():
    #iterating over sounds
    for sname in dataDetails[cname].keys():
      ftrArr.append(convFtrDict2List(dataDetails[cname][sname]['feature'])[descInput])
      infoArr.append([sname, cname])
  
  ftrArr = np.array(ftrArr)
  infoArr = np.array(infoArr)
  
  ftrArrWhite = whiten(ftrArr)
  centroids, distortion = kmeans(ftrArrWhite, nCluster)
  clusResults = -1*np.ones(ftrArrWhite.shape[0])
  
  for ii in range(ftrArrWhite.shape[0]):
    diff = centroids - ftrArrWhite[ii,:]
    diff = np.sum(np.power(diff,2), axis = 1)
    indMin = np.argmin(diff)
    clusResults[ii] = indMin
  
  ClusterOut = []
  classCluster = []
  globalDecisions = []  
  for ii in range(nCluster):
    ind = np.where(clusResults==ii)[0]
    freqCnt = []
    for elem in infoArr[ind,1]:
      freqCnt.append(infoArr[ind,1].tolist().count(elem))
    indMax = np.argmax(freqCnt)
    classCluster.append(infoArr[ind,1][indMax])
    
    print("\n(Cluster: " + str(ii) + ") Using majority voting as a criterion this cluster belongs to " + 
          "class: " + classCluster[-1])
    print ("Number of sounds in this cluster are: " + str(len(ind)))
    decisions = []
    for jj in ind:
        if infoArr[jj,1] == classCluster[-1]:
            decisions.append(1)
        else:
            decisions.append(0)
    globalDecisions.extend(decisions)
    print ("sound-id, sound-class, classification decision")
    ClusterOut.append(np.hstack((infoArr[ind],np.array([decisions]).T)))
    print (ClusterOut[-1])
  globalDecisions = np.array(globalDecisions)
  totalSounds = len(globalDecisions)
  nIncorrectClassified = len(np.where(globalDecisions==0)[0])
  print("Out of %d sounds, %d sounds are incorrectly classified considering that one cluster should "
        "ideally contain sounds from only a single class"%(totalSounds, nIncorrectClassified))
  print("You obtain a classification (based on obtained clusters and majority voting) accuracy "
         "of %.2f percentage"%round(float(100.0*float(totalSounds-nIncorrectClassified)/totalSounds),2))
  # return ClusterOut

In [ ]:
DESCRIPTOR_MAPPING = globals().get('descriptorMapping')
if DESCRIPTOR_MAPPING is None:
    raise NameError("descriptorMapping is not defined. Run Cell 1 before running this cell.")

def fetch_data_details(input_dir: str, desc_ext: str = '.json'):
    return fetchDataDetails(input_dir, desc_ext)

def _collect_sound_files(base_dir: Path) -> list[Path]:
    files = []
    for extension in ('*.mp3', '*.ogg', '*.wav'):
        files.extend(sorted(base_dir.rglob(extension)))
    return files

def _project_to_2d_pca(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if data.ndim != 2:
        raise ValueError('PCA input data must be a 2D array')

    if data.shape[1] == 1:
        projected = np.column_stack((data[:, 0], np.zeros(data.shape[0])))
        components = np.array([[1.0], [0.0]])
        explained_variance_ratio = np.array([1.0, 0.0])
        return projected, components, explained_variance_ratio

    centered = data - np.mean(data, axis=0, keepdims=True)
    _, singular_values, vt = np.linalg.svd(centered, full_matrices=False)
    components = vt[:2]
    projected = centered @ components.T

    variances = (singular_values ** 2) / max(data.shape[0] - 1, 1)
    total_variance = np.sum(variances)
    if total_variance == 0:
        explained_variance_ratio = np.zeros(2)
    else:
        explained_variance_ratio = variances[:2] / total_variance

    if explained_variance_ratio.shape[0] < 2:
        explained_variance_ratio = np.pad(explained_variance_ratio, (0, 2 - explained_variance_ratio.shape[0]))

    return projected, components, explained_variance_ratio

def _plot_clusters_interactive(
    data_array: np.ndarray,
    white_feature_array: np.ndarray,
    cluster_indices: np.ndarray,
    centroids: np.ndarray,
    sound_names: list[str],
    desc_input: list[int],
    n_cluster: int,
    sound_files: list[Path] = None
) -> plt.Figure:
    """Plot clusters with interactive click to show sound info and play sounds."""

    backend = matplotlib.get_backend().lower()
    if 'inline' in backend:
        print("\nInteractive clicks are limited with the inline backend.")
        print("Use `%matplotlib widget` (preferred) or `%matplotlib notebook`, then re-run this cell.")

    projected_points, pca_components, explained_variance_ratio = _project_to_2d_pca(white_feature_array)
    projected_centroids = (centroids - np.mean(white_feature_array, axis=0, keepdims=True)) @ pca_components.T
    print(
        f"\nPCA projection explained variance: "
        f"PC1={explained_variance_ratio[0] * 100:.1f}% , PC2={explained_variance_ratio[1] * 100:.1f}%"
    )

    fig, ax = plt.subplots(figsize=(12, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, n_cluster))
    point_to_index: dict[Any, np.ndarray] = {}

    for i in range(n_cluster):
        mask = cluster_indices == i
        cluster_data = projected_points[mask]

        if len(cluster_data) > 0:
            points = ax.scatter(
                cluster_data[:, 0],
                cluster_data[:, 1],
                c=[colors[i % len(colors)]],
                label=f'Cluster {i} ({mask.sum()} sounds)',
                alpha=0.6,
                s=100,
                edgecolors='black',
                linewidth=1,
                picker=True,
                pickradius=8,
            )
            point_to_index[points] = np.where(mask)[0]

    ax.scatter(
        projected_centroids[:, 0],
        projected_centroids[:, 1],
        c='red',
        marker='*',
        s=500,
        edgecolors='black',
        linewidth=2,
        label='Centroids',
        zorder=5
    )

    distances = np.zeros((len(sound_names), n_cluster))
    for i in range(n_cluster):
        distances[:, i] = np.linalg.norm(white_feature_array - centroids[i], axis=1)


    ax.set_xlabel(f"PC1 ({explained_variance_ratio[0] * 100:.1f}% variance)", fontsize=12)
    ax.set_ylabel(f"PC2 ({explained_variance_ratio[1] * 100:.1f}% variance)", fontsize=12)
    ax.set_title(f'K-means Clustering (k={n_cluster}) with PCA Projection\nClick on points to see details', fontsize=14)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

    annot = ax.annotate(
        "",
        xy=(0, 0),
        xytext=(20, 20),
        textcoords="offset points",
        bbox=dict(boxstyle="round", fc="yellow", alpha=0.9),
        arrowprops=dict(arrowstyle="->")
    )
    annot.set_visible(False)

    playback_process = None

    def play_sound(sound_name: str):
        nonlocal playback_process
        if not sound_files or len(sound_files) == 0:
            print("No sound files available to play")
            return

        print(f"\nLooking for sound file: {sound_name}")

        matching_files = [f for f in sound_files if f.parent.name == sound_name]
        if not matching_files:
            matching_files = [f for f in sound_files if f.stem == sound_name]

        if matching_files:
            sound_file = matching_files[0]
            print(f"Found: {sound_file}")

            try:
                if playback_process is not None and playback_process.poll() is None:
                    playback_process.terminate()
                if platform.system() == 'Darwin':
                    playback_process = subprocess.Popen(['afplay', str(sound_file)])
                elif platform.system() == 'Linux':
                    playback_process = subprocess.Popen(['aplay', str(sound_file)])
                elif platform.system() == 'Windows':
                    playback_process = subprocess.Popen(['start', str(sound_file)], shell=True)
                else:
                    print(f"Audio playback not configured for this OS. File: {sound_file}")
                    return
                print(f"Playing: {sound_name}")
            except Exception as error:
                print(f"Error playing sound: {error}")
        else:
            print(f"Sound file not found for: {sound_name}")
            print(f"Available files: {[f.parent.name for f in sound_files[:5]]}")

    def show_point_info(closest_idx: int):
        cluster = cluster_indices[closest_idx]
        sound_name = sound_names[closest_idx]
        play_sound(sound_name)

        dist_to_assigned = distances[closest_idx, cluster]

        info_lines = [
            f"Sound: {sound_name}",
            f"Assigned: Cluster {cluster}",
            f"Distance to centroid: {dist_to_assigned:.3f}"
        ]

        if n_cluster > 1:
            nearest_two = np.sort(distances[closest_idx])[:2]
            info_lines.append(f"Gap to next centroid: {nearest_two[1] - nearest_two[0]:.3f}")

        info_lines.append("\nDescriptor values:")
        for i, desc_idx in enumerate(desc_input[:5]):
            info_lines.append(f"{DESCRIPTOR_MAPPING[desc_idx]}: {data_array[closest_idx, i]:.3f}")

        annot.xy = (projected_points[closest_idx, 0], projected_points[closest_idx, 1])
        annot.set_text("\n".join(info_lines))
        annot.set_visible(True)

        ax.plot(
            projected_points[closest_idx, 0],
            projected_points[closest_idx, 1],
            'o',
            markersize=15,
            markerfacecolor='none',
            markeredgecolor='red',
            markeredgewidth=3
        )
        fig.canvas.draw_idle()

    def on_pick(event):
        artist = event.artist
        if artist not in point_to_index or len(event.ind) == 0:
            return
        cluster_global_indices = point_to_index[artist]
        closest_idx = int(cluster_global_indices[event.ind[0]])
        show_point_info(closest_idx)

    def on_click(event):
        if event.inaxes != ax or event.xdata is None or event.ydata is None:
            return

        min_dist = float('inf')
        closest_idx = 0

        for idx in range(len(sound_names)):
            point_x = projected_points[idx, 0]
            point_y = projected_points[idx, 1]
            dist = np.sqrt((event.xdata - point_x) ** 2 + (event.ydata - point_y) ** 2)
            if dist < min_dist:
                min_dist = dist
                closest_idx = idx

        show_point_info(closest_idx)

    def on_key(event):
        if event.key == 'escape':
            annot.set_visible(False)
            fig.canvas.draw_idle()

    fig.canvas.mpl_connect('pick_event', on_pick)
    fig.canvas.mpl_connect('button_press_event', on_click)
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    return fig

def run_interactive_clustering(
    target_dir: str,
    n_cluster: int = -1,
    desc_input: list[int] | None = None,
    show_plot: bool = True,
    seed: int = 42
) -> plt.Figure:
    if desc_input is None:
        desc_input = []

    if len(desc_input) == 0:
        raise ValueError('desc_input must contain at least one descriptor index')

    data_details = fetch_data_details(target_dir)
    if not data_details:
        raise ValueError(f"No descriptor files in {Path(target_dir).resolve()}. Run the data setup cell first, or point DATA_DIR to a populated cache.")
    if n_cluster == -1:
        n_cluster = len(data_details.keys())

    feature_array = []
    sound_names = []
    search_categories = []
    for class_name in data_details.keys():
        for sound_name in data_details[class_name].keys():
            full_vector = convFtrDict2List(data_details[class_name][sound_name]['feature'])
            feature_array.append(full_vector[desc_input])
            sound_names.append(sound_name)
            search_categories.append(class_name)

    data_array = np.array(feature_array)
    scale = data_array.std(axis=0)
    white_feature_array = data_array / np.where(scale > 0, scale, 1)
    distinct_count = len(np.unique(white_feature_array, axis=0))
    if not 1 <= n_cluster <= distinct_count:
        raise ValueError(f"Choose between 1 and {distinct_count} clusters for this dataset.")
    centroids, _ = kmeans(white_feature_array, n_cluster, seed=seed)
    from scipy.cluster.vq import vq

    cluster_indices, _ = vq(white_feature_array, centroids)
    if len(centroids) < n_cluster:
        print(f"K-means returned {len(centroids)} nonempty centroids for k={n_cluster}.")
    n_cluster = len(centroids)

    sound_files = _collect_sound_files(Path(target_dir))
    fig = _plot_clusters_interactive(
        data_array=data_array,
        white_feature_array=white_feature_array,
        cluster_indices=cluster_indices,
        centroids=centroids,
        sound_names=sound_names,
        desc_input=desc_input,
        n_cluster=n_cluster,
        sound_files=sound_files,
    )

    if show_plot:
        plt.show()
        plot_cluster_summary(search_categories, cluster_indices, centroids,
                             white_feature_array.mean(axis=0), desc_input)
        plt.show()
    return fig

def compute_similar_sounds(query_file: str, target_dir: str, desc_input: list[int] | None = None):
    if desc_input is None:
        desc_input = []

    data_details = fetch_data_details(target_dir)
    with Path(query_file).open("r", encoding="utf-8") as file_obj:
        query_features = json.load(file_obj)

    distance_list = []
    for class_name in data_details.keys():
        for sound_name in data_details[class_name].keys():
            f1 = convFtrDict2List(query_features)
            f2 = convFtrDict2List(data_details[class_name][sound_name]["feature"])
            euclidean_distance = np.sqrt(np.sum(np.power(np.array(f1[desc_input]) - np.array(f2[desc_input]), 2)))
            distance_list.append([euclidean_distance, sound_name, class_name])

    sorted_indices = np.argsort(np.array(distance_list)[:, 0])
    return (np.array(distance_list)[sorted_indices, :]).tolist()

def classify_sound_knn(query_file: str, target_dir: str, k_neighbors: int, desc_input: list[int] | None = None):
    if desc_input is None:
        desc_input = []

    distances = compute_similar_sounds(query_file, target_dir, desc_input)
    if len(np.where((np.array(distances)[:, 0].astype(np.float64)) == 0)[0]) > 0:
        print(
            "Warning: exact copy of the query file found in target directory."
            " Check for duplicates during KNN classification."
        )

    classes = np.array(distances)[:k_neighbors, 2].tolist()
    frequency_count = []
    for idx in range(k_neighbors):
        frequency_count.append(classes.count(classes[idx]))

    max_index = np.argmax(frequency_count)
    predicted_class = classes[max_index]
    print(f"This sample belongs to class: {predicted_class}")
    return predicted_class

def plot_cluster_summary(search_categories, cluster_indices, centroids, feature_means, desc_input):
    categories, category_codes = np.unique(search_categories, return_inverse=True)
    counts = np.zeros((len(centroids), len(categories)), dtype=int)
    np.add.at(counts, (cluster_indices, category_codes), 1)
    profiles = centroids - feature_means
    limit = max(np.abs(profiles).max(), 1e-12)
    labels = [f'Cluster {i}' for i in range(len(centroids))]
    descriptor_labels = [DESCRIPTOR_MAPPING[i].removeprefix('lowlevel.').removeprefix('sfx.')
                         for i in desc_input]

    fig, (composition, profile) = plt.subplots(2, 1, figsize=(max(9, len(desc_input) * .62), 7),
                                                layout='constrained')
    image = composition.imshow(counts, aspect='auto', cmap='Blues')
    for (row, col), count in np.ndenumerate(counts):
        composition.text(col, row, str(count), ha='center', va='center',
                         color='white' if count > counts.max() / 2 else 'black')
    composition.set(xticks=np.arange(len(categories)), xticklabels=categories,
                    yticks=np.arange(len(labels)), yticklabels=labels,
                    xlabel='Freesound search category', title='Recordings per cluster and search category')
    fig.colorbar(image, ax=composition, label='Number of recordings')

    image = profile.imshow(profiles, aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit)
    profile.set(xticks=np.arange(len(desc_input)), xticklabels=descriptor_labels,
                yticks=np.arange(len(labels)), yticklabels=labels,
                title='Cluster centroid relative to the collection mean')
    plt.setp(profile.get_xticklabels(), rotation=45, ha='right')
    fig.colorbar(image, ax=profile, label='Difference in standard deviations')

In [40]:
# Interactive backend setup for click-to-play plots
ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic('matplotlib', 'widget')
        print('Using matplotlib widget backend')
    except Exception as error_widget:
        print(f'widget backend unavailable: {error_widget}')
        try:
            ip.run_line_magic('matplotlib', 'notebook')
            print('Using matplotlib notebook backend')
        except Exception as error_notebook:
            print(f'notebook backend unavailable: {error_notebook}')
            ip.run_line_magic('matplotlib', 'inline')
            print('Falling back to inline backend (click events may not work).')

Using matplotlib widget backend


## 3. Fit once and inspect the clusters

The default uses five clusters for five search categories. The fit uses all selected descriptors. The PCA plot places the same sounds and centroids on two axes and reports how much descriptor variance those axes capture. Apparent overlap in two dimensions can conceal separation along omitted directions.

- **PCA and listening**. With an interactive backend, click a sound to hear its cached preview and inspect its distance to its assigned centroid. A small gap to the next centroid indicates two similarly close centers in the full feature space.
- **Search-category counts**. Read across a row to see what one cluster contains. Read down a column to see how a search category splits across clusters. Several clusters can share the same dominant category.
- **Centroid profiles**. Red means the cluster center is above the collection mean for that descriptor. Blue means it is below. Compare the strongest differences with the recordings you hear.

Try changing `N_CLUSTERS` from five to three. Which sounds end up together, and which descriptors help explain the change?

In [ ]:
N_CLUSTERS = 5
FEATURE_INDICES = list(descriptorMapping)

_ = run_interactive_clustering(
    target_dir=str(DATA_DIR),
    n_cluster=N_CLUSTERS,
    desc_input=FEATURE_INDICES,
    show_plot=True,
)